In [ ]:
install.packages("rio")

install.packages("sqldf")

install.packages("tidyverse")

install.packages("tibble")

In [ ]:
library(rio)

library(sqldf)

library(tidyverse)

library(tibble)

In [ ]:
list.files("/content")

In [ ]:
app_events <- import("/content/clean_app_events.csv")

complaints <- import("/content/clean_complaints.csv")

customers <- import("/content/clean_customers.csv")

deliveries <- import("/content/clean_deliveries.csv")

drivers <- import("/content/clean_drivers.csv")

hubs <- import("/content/clean_hubs.csv")

incidents <- import("/content/clean_incidents.csv")

orders <- import("/content/clean_orders.csv")

vehicles <- import("/content/clean_vehicles.csv")

print("All datasets loaded successfully")

In [ ]:
head(orders)

head(deliveries)

head(customers)

head(drivers)

head(complaints)

head(hubs)

head(app_events)

head(incidents)

head(vehicles)


In [ ]:
str(orders)

str(deliveries)

str(customers)

str(drivers)

str(complaints)

str(hubs)

str(app_events)

str(incidents)

str(vehicles)


In [ ]:
summary(orders)

summary(deliveries)

summary(complaints)

summary(vehicles)

In [ ]:
summary_data <- orders %>%
  summarise(
    average_order_value = mean(order_value, na.rm = TRUE),
    maximum_order_value = max(order_value, na.rm = TRUE),
    minimum_order_value = min(order_value, na.rm = TRUE)
  )

print(summary_data)

In [ ]:
sqldf("
SELECT COUNT(*) AS total_orders
FROM orders
")


In [ ]:
sqldf("
SELECT delivery_status,
COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY delivery_status
")

In [ ]:
sqldf("
SELECT driver_id,
COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY driver_id
ORDER BY total_deliveries DESC
LIMIT 10
")

In [ ]:
sqldf("
SELECT complaint_type,
COUNT(*) AS total_complaints
FROM complaints
GROUP BY complaint_type
")

In [ ]:
sqldf("
SELECT hub_id,
COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY hub_id
")

In [ ]:
sqldf("
SELECT customer_id,
COUNT(*) AS total_orders
FROM orders
GROUP BY customer_id
LIMIT 10
")

In [ ]:
sqldf("
SELECT incident_type,
COUNT(*) AS total_incidents
FROM incidents
GROUP BY incident_type
")

In [ ]:
sqldf("
SELECT order_id,
customer_id,
order_value
FROM orders
WHERE order_value > 100
ORDER BY order_value DESC
")

In [ ]:
sqldf("
SELECT event_type,
COUNT(*) AS total_events
FROM app_events
GROUP BY event_type
")

In [ ]:
transformed_orders <- orders %>%
  mutate(
    increased_order_value = order_value * 1.10
  )

head(transformed_orders)


In [ ]:
ggplot(deliveries, aes(x = delivery_status)) +
  geom_bar() +
  labs(
    title = "Delivery Status Distribution",
    x = "Delivery Status",
    y = "Count"
  )

In [ ]:
ggplot(complaints, aes(x = complaint_type)) +
  geom_bar() +
  labs(
    title = "Complaint Type Distribution",
    x = "Complaint Type",
    y = "Count"
  )

In [ ]:
ggplot(vehicles, aes(x = vehicle_type)) +
  geom_bar() +
  labs(
    title = "Vehicle Type Distribution",
    x = "Vehicle Type",
    y = "Count"
  )

In [ ]:
deliveries %>%
  count(delivery_status) %>%
  ggplot(aes(x = "", y = n, fill = delivery_status)) +
  geom_bar(stat = "identity", width = 1) +
  coord_polar("y", start = 0) +
  labs(title = "Delivery Status Pie Chart")

In [ ]:
ggplot(
  orders,
  aes(
    x = promised_window_hours,
    y = order_value
  )
) +
  geom_point() +
  labs(
    title = "Order Value vs Promised Window Hours",
    x = "Promised Window Hours",
    y = "Order Value"
  )

In [ ]:
if(
  "order_value" %in% colnames(orders) &
  "promised_window_hours" %in% colnames(orders)
){

  correlation_test <- cor.test(
    orders$order_value,
    orders$promised_window_hours
  )

  print(correlation_test)
}



In [ ]:
if(
  "order_value" %in% colnames(orders) &
  "promised_window_hours" %in% colnames(orders)
){

  regression_model <- lm(
    order_value ~ promised_window_hours,
    data = orders
  )

  summary(regression_model)
}


In [ ]:
if(exists("regression_model")){

  predictions <- predict(
    regression_model,
    newdata = orders
  )

  print(head(predictions))
}


In [ ]:
if(exists("predictions")){

  mse <- mean(
    (orders$order_value - predictions)^2,
    na.rm = TRUE
  )

  print(mse)
}


In [ ]:
merged_data <- sqldf("
SELECT
o.order_id,
o.customer_id,
o.order_value,
d.delivery_id,
d.driver_id,
d.hub_id,
d.delivery_status
FROM orders o
LEFT JOIN deliveries d
ON o.order_id = d.order_id
")

In [ ]:
head(merged_data)

str(merged_data)


In [ ]:
export(
  merged_data,
  "/content/merged_northstar_data.csv"
)

print("Merged Dataset Saved Successfully")



In [ ]:
print("R Analytics and SQL Analysis Completed Successfully")